# 3-2. 재순위화 (Reranking)

## 학습 목표
- Bi-encoder와 Cross-encoder의 구조·속도·품질 차이를 설명할 수 있다.
- BGE-reranker-v2-m3(🔒)를 사용해 기존 retriever의 상위 K 결과를 재정렬할 수 있다.
- 이전 노트북의 **하이브리드(BM25 + Dense)** 를 1단계로 쓰고, 그 위에 reranker를 얹어 **완성형 2-stage retrieval** 파이프라인을 구현할 수 있다.
- 전자금융 표준약관 벤치마크 5개 질문으로 **hybrid vs hybrid+rerank** 결과를 비교·평가할 수 있다.

## 핵심 키워드
Bi-encoder · Cross-encoder · BGE-reranker-v2-m3 · 2-stage retrieval · Hybrid + Rerank · NDCG

## 이 노트북의 위치

```
1단계 (이전 노트북 01)              2단계 (이 노트북)
Hybrid (Dense + BM25, RRF)   ─▶   Cross-Encoder Reranker
└─ "넓게 후보를 모은다"              └─ "정답을 꼭대기에 올린다"
       recall 확보                      precision 확보
```

`01_hybrid_ensemble.ipynb` 에서 만든 **하이브리드 retriever가 이 노트북의 1단계 입력**이다. 여기서는 그 상위 K 후보를 Cross-Encoder로 다시 채점해 최종 top-N을 골라낸다. Anthropic Contextual Retrieval 실험(2024)에서 dense 단독 대비 **실패율 67% 감소**로 보고된 바로 그 구성이다.

## 사전 준비
- `data/corpus_ko.txt` 로부터 하이브리드 retriever(FAISS + BM25)와 reranker를 **본 노트북 내에서 즉석 구축**한다. 01 노트북을 먼저 실행하지 않아도 독립적으로 돌아간다.

In [ ]:
# 공통 세팅: common 헬퍼를 사용하기 위해 repo 루트를 sys.path에 추가
import sys, os
sys.path.insert(0, '../..')

from common import get_chat_model, get_embeddings, provider_badge
from common.ko_tokenizer import tokenize_ko
from common.loaders import load_any

print(provider_badge())


## 1. 왜 재순위화가 필요한가?

1단계 검색기(하이브리드든 dense 단독이든)는 공통적으로 **질문과 문서를 각자 따로 인코딩**한다. BM25는 토큰 빈도로, dense는 벡터 유사도로 점수를 매기지만 둘 다 *질문과 문서 토큰 간의 상호작용* 을 보지 않는다. 아래 두 인코딩 방식의 차이를 먼저 짚어보자.

### Bi-encoder (🔒 빠름, 품질 중간)
```
질문 ─▶ [Encoder] ─▶ q_vec ┐
                              ├─ cosine ─▶ score
문서 ─▶ [Encoder] ─▶ d_vec ┘
```
- 문서 임베딩을 **오프라인에서 미리** 계산해 저장 → 대규모 검색에 유리.
- 질의 시 1회의 인코딩 + 수백만 건의 벡터 내적(ANN 인덱스로 O(log N) 수준)만 수행.
- 단점: 질문과 문서를 독립적으로 인코딩하므로 **상호작용 정보**가 없다. "망분리"라는 단어가 겹치기만 해도 높은 점수를 얻을 수 있어 **표면적 유사도**에 취약하다.

### Cross-encoder (🔒 느림, 품질 높음)
```
[CLS] 질문 [SEP] 문서 [SEP] ─▶ [Encoder] ─▶ 관련성 score (스칼라)
```
- 질문과 문서를 **한 시퀀스로 연결해 함께** 인코딩 → 모든 토큰 쌍 간 self-attention으로 세밀한 의미 매칭이 가능.
- "질문이 묻는 **무엇**"과 "문서가 말하는 **무엇**"을 토큰 레벨에서 비교하므로, 단순 키워드 중복이 아니라 *실제 답을 담고 있는지*를 판별할 수 있다.
- 단점: 모든 후보 (질문, 문서) 쌍에 대해 모델 forward를 1번씩 돌려야 해서 비용이 O(K). 100만 건 전체에 적용하는 것은 비현실적.

### 실무 해결: 2-stage retrieval

| 단계 | 방식 | 후보 수 | 속도 | 역할 |
|------|------|--------|------|------|
| 1단계 | **Hybrid (Dense + BM25 + RRF)** | top-20~100 | 빠름 | **recall** 확보 (정답이 후보에 "들어오게") |
| 2단계 | Cross-encoder (rerank) | top-5~10 | 느림 | **precision** 확보 (정답을 "위로 올리기") |

> 💡 **직관**: 1단계는 "관련 있어 보이는 20~100개를 빠르게 긁어오고", 2단계는 "그중 정말 관련 있는 5~10개를 꼼꼼히 고른다". 넓은 그물로 물고기를 퍼올린 뒤(recall) 상품 가치가 있는 것만 고르는(precision) 어시장 방식과 같다.

> 📌 **왜 1단계를 하이브리드로 쓰는가?** Reranker는 *이미 검색된 후보만* 재정렬할 수 있다. Dense 단독은 "전자금융감독규정" 같은 **고유명사·희귀 전문용어**를 놓칠 수 있는데, 이렇게 놓친 정답은 reranker가 아무리 강력해도 복구할 수 없다. BM25와의 하이브리드는 이 누락을 막아 reranker에게 **재정렬할 가치가 있는 후보 풀**을 안정적으로 공급한다.

### 언제 reranker를 쓰지 말아야 하는가?
- **LLM 컨텍스트가 충분하고 top-5 정확도가 이미 높다면** → 추가 지연을 감수할 이유가 적다.
- **실시간성이 극단적으로 중요한 경우** (예: 자동완성) → 100ms의 rerank 비용이 부담.
- **후보 문서가 매우 길다면** → cross-encoder 입력 길이 제한(보통 512 토큰)으로 잘려서 이점이 사라질 수 있다. 청크 단위가 적절한지 먼저 점검.


## 2. 1단계 하이브리드 retriever 구축

`data/corpus_ko.txt` 를 로드해 1단계 recall 확보용 **하이브리드 retriever**(FAISS dense + BM25 sparse, RRF 결합)를 구성한다. 이 구성은 이전 노트북 `01_hybrid_ensemble.ipynb` 에서 다룬 패턴을 그대로 가져온 것이다.

- `chunk_size=300, chunk_overlap=50`: 한국어 법령·규정 문장 기준으로 한두 문장 단위의 의미 단위를 유지하는 값. 너무 크면 rerank 입력이 늘어 속도 손해, 너무 작으면 맥락이 잘린다.
- `k=20`: 2단계 rerank가 충분한 후보 풀을 보도록 1단계에서 **과잉 검색(over-retrieve)** 한다. 최종 출력이 top-5라면 1단계는 최소 그 **4배 이상**을 뽑는 것이 관례(Anthropic 실험에서는 top-150 → top-20).
- `tokenize_ko` (kiwi 형태소 분석기): 한국어 BM25에서 "전자금융거래는/가/의"를 같은 어간으로 묶기 위한 필수 전처리.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

# 코퍼스 로드 및 청킹
docs = load_any('../../data/corpus_ko.txt')
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(f'전체 청크 수: {len(chunks)}')

# (A) Dense retriever (FAISS)
vectordb = FAISS.from_documents(chunks, get_embeddings())
dense = vectordb.as_retriever(search_kwargs={'k': 20})

# (B) Sparse retriever (BM25 + kiwi 형태소)
sparse = BM25Retriever.from_documents(chunks, preprocess_func=tokenize_ko)
sparse.k = 20

# (C) 하이브리드: Dense + BM25를 RRF로 결합 → 1단계 base_retriever
base_retriever = EnsembleRetriever(
    retrievers=[dense, sparse],
    weights=[0.5, 0.5],
)
print('1단계 하이브리드 retriever 구축 완료 (FAISS + BM25, top-20)')


## 3. BGE-reranker-v2-m3 로드 🔒

[BAAI/bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3)는 한국어·영어·중국어를 포함한 100+개 언어를 지원하는 다국어 cross-encoder 모델이다. 이름의 `m3`는 **Multi-Linguality, Multi-Functionality, Multi-Granularity** 를 의미하며, 문장/문단/문서 어느 단위에서도 안정적으로 동작하도록 학습됐다.

### 핵심 특징
- **XLM-RoBERTa large 기반** (~568M 파라미터): GPU 없이도 CPU/FP16으로 실행 가능.
- **입력 길이 8192 토큰**: 일반 cross-encoder(512)보다 길어 법령·약관 같은 긴 청크에 유리.
- `normalize=True` 옵션: raw logit을 시그모이드로 0~1 범위 확률 유사 점수로 변환 → 임계값 설정이 쉬움.

### 점수 해석 가이드 (경험칙)
| 점수 범위 | 의미 |
|----------|------|
| > 0.8 | 질문에 직접적 답을 포함 (높은 신뢰) |
| 0.4 ~ 0.8 | 관련은 있으나 부분적·간접적 |
| < 0.4 | 잡음에 가까움 — 상위 K에서 제외 고려 |

> ⚠️ 최초 실행 시 약 2.3GB 모델을 다운로드한다. 폐쇄망에서는 `HF_HOME`에 사전 반입이 필요하다. 사내 모델 허브가 있다면 `local_files_only=True` 로 로컬 경로를 지정해 쓸 수 있다.


In [ ]:
from FlagEmbedding import FlagReranker

# normalize=True로 설정하면 0~1 사이 확률 유사 점수로 변환
reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)

# 간단 테스트: 질문-문서 페어에 대한 점수
pairs = [
    ('망분리란 무엇인가?', '망분리는 업무망과 인터넷망을 분리해 보안을 강화한다.'),
    ('망분리란 무엇인가?', 'LLM은 Transformer 아키텍처 기반의 모델이다.'),
]
scores = reranker.compute_score(pairs, normalize=True)
for (q, d), s in zip(pairs, scores):
    print(f'score={s:.4f} | {d[:40]}...')


## 4. LangChain ContextualCompressionRetriever에 reranker 연결

LangChain은 "1단계 검색 → 후처리(압축/필터링/재정렬)" 패턴을 **`ContextualCompressionRetriever`** 로 추상화해 제공한다. 이름은 "compression"이지만 실제로는 *후보 집합을 줄이거나 재배열하는 모든 연산* 을 포괄한다 — reranking도 그중 하나다.

```
사용자 query
    │
    ▼
base_retriever  ──▶ top-20 후보
  ├─ FAISS (dense)
  └─ BM25  (sparse)  ← RRF로 결합
    │
    ▼
base_compressor (BGE reranker) ──▶ top-5 재정렬된 결과
```

`BaseDocumentCompressor`를 상속해 `compress_documents(documents, query)` 메서드만 구현하면 된다. 이 인터페이스 덕분에 reranker를 다른 LangChain 체인(RAG, agent 등)에 **투명하게 끼워 넣을 수 있다**. 재순위 점수는 `doc.metadata['rerank_score']` 에 기록해 두면 디버깅·모니터링에 유용하다.

In [ ]:
from typing import Sequence
from langchain_core.documents import Document
from langchain_core.callbacks import Callbacks
from langchain.retrievers import ContextualCompressionRetriever
from langchain_core.documents.compressor import BaseDocumentCompressor

class BgeRerankCompressor(BaseDocumentCompressor):
    '''BGE reranker를 LangChain compressor 인터페이스로 감싼 클래스.'''
    model_name: str = 'BAAI/bge-reranker-v2-m3'
    top_n: int = 5

    class Config:
        arbitrary_types_allowed = True

    def compress_documents(
        self,
        documents: Sequence[Document],
        query: str,
        callbacks: Callbacks = None,
    ) -> Sequence[Document]:
        if not documents:
            return []
        pairs = [(query, d.page_content) for d in documents]
        scores = reranker.compute_score(pairs, normalize=True)
        ranked = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
        top = ranked[: self.top_n]
        # 점수를 메타데이터에 기록
        out = []
        for d, s in top:
            d.metadata['rerank_score'] = float(s)
            out.append(d)
        return out

compressor = BgeRerankCompressor(top_n=5)
rerank_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,
)
print('2-stage retriever 준비 완료 (hybrid top-20 → rerank top-5)')


## 5. 벤치마크 5개 질문으로 hybrid vs hybrid+rerank 비교

이제 reranker가 **하이브리드 검색 결과 위에서도 추가 이득**을 주는지 정량적으로 검증한다. 단순히 dense 단독과 비교하는 것은 공정하지 않다 — 현실에서 1단계는 이미 하이브리드일 가능성이 높으니, **hybrid vs hybrid+rerank** 로 비교해야 "reranker의 마지막 1마일 효과"를 제대로 측정할 수 있다.

전자금융 표준약관 벤치마크 질문 5개를 노트북 내에 인라인으로 정의하고, 각 질문마다 **정답 키워드(gold keywords)** 를 지정한다.

### 평가 방식: Hit Rate @ K
- 상위 K개 검색 결과에 정답 키워드가 얼마나 포함됐는지를 비율로 측정.
- 간단하지만 직관적인 메트릭으로, reranker의 효과를 빠르게 체감할 수 있다.
- 실제 운영에서는 **NDCG@K**(순위 가중 관련도), **MRR**(첫 정답의 역순위), **Recall@K** 등을 병행 사용한다.

### 비교 지점
아래 코드에서는 동일한 질문에 대해 두 파이프라인 결과를 비교한다.
- **hybrid top-5**: FAISS + BM25 앙상블만으로 바로 자른 상위 5건 (baseline, 이미 강력한 1단계)
- **hybrid+rerank top-5**: hybrid top-20 → BGE reranker top-5 (완성형 2-stage)

> 📌 **무엇을 봐야 하나**:
> - **개선 폭이 0에 가까운 질문**: 하이브리드가 이미 정답을 상위에 올린 경우. Reranker의 한계 효용이 낮은 구간.
> - **큰 양수 개선**: 하이브리드는 정답을 후보 20개 안에 "넣어주긴" 했지만 10위권에 묻어뒀고, reranker가 이를 top-5로 끌어올린 경우. 여기서 reranker의 진짜 가치가 드러난다.
> - **음수(악화)**: Reranker가 표면적으로 유사해 보이는 잡음 문서를 잘못 올린 경우. 후보 풀 크기(K=20)나 청크 전략 재검토가 필요하다.

In [ ]:
BENCHMARK = [
    {
        'q': '전자금융거래의 정의는 무엇인가?',
        'gold_keywords': ['전자적 장치', '자동화된 방식', '금융회사'],
    },
    {
        'q': '개인정보보호법에서 개인정보는 어떻게 정의되는가?',
        'gold_keywords': ['살아 있는 개인', '성명', '주민등록번호'],
    },
    {
        'q': '금융회사가 신용정보를 처리할 때 준수해야 하는 원칙은?',
        'gold_keywords': ['신용정보의 이용 및 보호', '최소한의 범위', '수탁자'],
    },
    {
        'q': '망분리란 무엇이며 왜 금융권에서 중요한가?',
        'gold_keywords': ['업무망', '인터넷망', '전자금융감독규정'],
    },
    {
        'q': 'RAG가 LLM 환각 문제에 어떻게 도움이 되는가?',
        'gold_keywords': ['검색', '컨텍스트', '환각'],
    },
]

def hit_rate(doc_texts: list[str], keywords: list[str]) -> float:
    '''상위 검색 결과에서 정답 키워드가 포함된 비율.'''
    joined = ' '.join(doc_texts)
    return sum(1 for k in keywords if k in joined) / len(keywords)


In [ ]:
import pandas as pd

rows = []
for item in BENCHMARK:
    q, kws = item['q'], item['gold_keywords']
    # 1단계 하이브리드만 (top-5로 잘라 비교)
    hybrid_docs = base_retriever.invoke(q)[:5]
    hybrid_hit = hit_rate([d.page_content for d in hybrid_docs], kws)
    # 2-stage: 하이브리드 top-20 → rerank top-5
    rerank_docs = rerank_retriever.invoke(q)
    rerank_hit = hit_rate([d.page_content for d in rerank_docs], kws)
    rows.append({
        '질문': q,
        'hybrid top-5 hit': f'{hybrid_hit:.2%}',
        'hybrid+rerank top-5 hit': f'{rerank_hit:.2%}',
        '개선': f'{(rerank_hit - hybrid_hit):+.2%}',
    })

df = pd.DataFrame(rows)
df
